This script prepares the input files for the BGO3 Year 2022+2023 Runs


#### Functions

In [1]:
### Libraries
import os
import glob
import fnmatch

import numpy as np
import xarray as xr
import pandas as pd
# import ESMF
import datetime

# get the packages downloaded from NCAR cheyenne: /glade/u/home/cdswk/python/lib/my_packages/
import sys
sys.path.insert(0,'/home/taoma528/Scripts/CESM_analysis/MergeNEI2017_to_CAMS/NCAR_packages/')

# from the module
from dsj.array.chk import chk
from dsj.plot.Plot_2D import Plot_2D
from dsj.analysis.Calc_Emis import Calc_Emis_T
# from dsj.analysis.Regridding_ESMF import Add_bounds, Regridding | ESMF still not working here, try on Casper 
from dsj.io.get_sp import get_sp

/home/taoma528/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
import os

def remove_file(file_path):
    """
    Remove a text file at the specified path.
    
    Args:
        file_path (str): The path to the text file.
    """
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"The file '{file_path}' has been successfully removed.")
    else:
        print(f"The file '{file_path}' does not exist.")

# e.g. remove_file('file.txt')


In [3]:
from dateutil.relativedelta import relativedelta

from datetime import datetime
def extract_date_and_time(filename):
    # Extract the date and time part of the filename
    date_time_str = filename[13:]
    # Convert the date and time string to a datetime object (without seconds)
    date_time_obj = datetime.strptime(date_time_str, '%Y-%m-%d_%H:%M:%S')
    return date_time_obj

# # Sort the list based on date and time using the custom key function
# sorted_filenames = sorted(filenames, key=extract_date_and_time)

In [4]:
import os
from netCDF4 import Dataset

def convert_netcdf4_to_classic(input_dir, output_dir):
    """
    Convert all NetCDF-4 files in the input directory to classic format and save them in the output directory.
    
    Parameters:
    input_dir (str): Directory containing NetCDF-4 files.
    output_dir (str): Directory to save the converted NetCDF classic files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Convert each .nc file
    for filename in os.listdir(input_dir):
        if filename.endswith('.nc'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)  # Keep the same filename
            
            try:
                # Open the NetCDF-4 file
                with Dataset(input_file_path, 'r') as src:
                    # Create a new NetCDF classic file
                    with Dataset(output_file_path, 'w', format='NETCDF3_CLASSIC') as dst:
                        # Copy dimensions
                        for name, dimension in src.dimensions.items():
                            dst.createDimension(name, (len(dimension) if not dimension.isunlimited() else None))
    
                        # Copy variables
                        for name, variable in src.variables.items():
                            dst.createVariable(name, variable.datatype, variable.dimensions)
                            dst[name][:] = src[name][:]
                            dst[name].setncatts({k: variable.getncattr(k) for k in variable.ncattrs()})
                
                print(f"Converted {input_file_path} to {output_file_path}")

            except Exception as e:
                print(f"Failed to convert {input_file_path}. Error: {e}")

# Example usage
# input_directory = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_netCDF4/'
# output_directory = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021/'

# convert_netcdf4_to_classic(input_directory, output_directory)


## Method 1- increase by 10 yrs - CMIP6: Aircraft_vertical/other emissions

In [56]:
### check the files to replace in nl file
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

import os
all_files = [f for f in os.listdir(target_diri) if os.path.isfile(os.path.join(target_diri, f))]
# print(all_files)

for filei in all_files:
    print(f'{target_diri}{filei}')


/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_SO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_DMS_other_surfto2025use2011to2015_ne30np4_c20200605.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_DMS_bb_surface_to2025use2011to2015_ne30np4_c20200606.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_num_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_NO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissi

In [60]:
### check the files to replace in nl file
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/'

import os
all_files = [f for f in os.listdir(target_diri) if os.path.isfile(os.path.join(target_diri, f))]
# print(all_files)

for filei in all_files:
    print(f'{target_diri}{filei}')
print(len(all_files))

/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_C3H6_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_CH3CHO_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_C2H4_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_C3H8_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_CH3OH_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_BENZENE_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/CAMS-GLOB-ANT_ne30np4_XYLENES_v6.2_monthly.nc
/net/fs09/d0/taoma528/cheyenne_copies/acom/MUS

In [59]:
import pandas as pd

start_date = pd.Timestamp("1950-01-01")
end_date = pd.Timestamp("2000-01-15")

delta_days = (end_date - start_date).days
print(delta_days)


18276


In [32]:
### Files to modify
# copy 2011 to 2021; 2012 to 2022; 2013 to 2023; 2014 to 2024;2015 to 2025
# - DMS -> emissions-cmip6_DMS_bb_surface_
# - 'DMS -> emissions-cmip6_DMS_other_surface_
# - 'num_a1 -> emissions-cmip6_num_so4_a1_bb_surface_

# Define directories and file list
sourceCopy_diri = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/'
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

filels = [
    'emissions-cmip6_DMS_bb_surface_175001-201512_ne30np4_c20200606.nc',
    'emissions-cmip6_DMS_other_surface_1750_2015_ne30np4_c20200605.nc',
    'emissions-cmip6_num_so4_a1_bb_surface_175001-201512_ne30np4_c20200606.nc',    
         ]

In [ ]:
# Check the format of a nc file
# ncdump -k /net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/emissions-cmip6_DMS_bb_surface_175001-201512_ne30np4_c20200606.nc

In [ ]:
# Loop through => except for time in cft format
cftimefiles = []
for fileidx in range(len(filels)):
    # Original file: CMIP6 emissions 
    filename = filels[fileidx]
    print(filename)
    
    # New filename to reflect copying 2011–2015 data to 2021–2025
    newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

    # Define year range to extract
    startDate = '2011-01-01'
    endDate = '2015-12-31'

    try:
        # Read original file for years 2011–2015
        filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

        ### Process
        # Step 1: Create new time coordinates by adding 10 years (shift 2011–2015 → 2021–2025)
        time_array_pd = pd.to_datetime(filei_da['time'].values)
        time_array_shifted = time_array_pd + pd.DateOffset(years=10)
        time_array_shifted_np = time_array_shifted.to_numpy()

        # Step 2: Duplicate the dataset and assign shifted time as float days since 1750-01-01
        filei_da_shifted = filei_da.copy(deep=True)

        # Reference date for new time encoding
        ref_date = pd.Timestamp('1750-01-01 00:00:00')
        # Shifted datetime
        time_array_shifted = pd.to_datetime(filei_da['time'].values) + pd.DateOffset(years=10)
        # Convert to float days since 1750-01-01
        days_since_ref = (time_array_shifted - ref_date) / pd.Timedelta(days=1)

        # Overwrite time as float array
        filei_da_shifted = filei_da_shifted.assign_coords(time=("time", days_since_ref.values.astype('float32')))
        filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
        filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'

        # Step 3: Add a new 'history' entry
        original_history = filei_da_shifted.attrs.get('history', '')
        new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2011–2015 data to 2021–2025 for LBC case study"
        combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
        filei_da_shifted.attrs['history'] = combined_history

        # Step 4: Shift date variable
        date_strings = filei_da_shifted['date'].values.astype(str)
        date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
        date_shifted = date_datetimes + pd.DateOffset(years=10)
        new_date_values = date_shifted.strftime('%Y%m%d').astype('int32')
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        # Replace the 'date' variable
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        ### Step 5: Save to new NetCDF file
        fileOUTpath = f'{target_diri}{newfilename}'
        filei_da_shifted.to_netcdf(fileOUTpath)
        print("Saved to:", fileOUTpath)
    
    except TypeError:
        cftimefiles.append(filename)
        print(f'cftime - {filename}')


In [47]:
# Loop through => ONLY for time in cft format
sourceCopy_diri = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/'
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

from datetime import datetime
import xarray as xr
import pandas as pd
import os

for fileidx in range(len(cftimefiles)):
    filename = cftimefiles[fileidx]
    
    # New filename reflects 10-year shift
    newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

    # Read in file
    filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

    ### Step 1: Extract original time (cftime) and convert to datetime
    time_cftime = filei_da['time'].values

    # Convert cftime objects to pandas datetime64 via ISO string conversion
    time_datetime = pd.to_datetime([t.isoformat() for t in time_cftime])

    # Step 2: Shift only years 2011–2015 by 10 years
    time_array_shifted = pd.to_datetime([
        t.replace(year=t.year + 10) if 2011 <= t.year <= 2015 else t
        for t in time_datetime
    ])

    # Normalize to midnight to avoid fractional days
    time_array_shifted = time_array_shifted.normalize()

    # Convert shifted datetime to float days since 1750-01-01
    ref_date = pd.Timestamp('1750-01-01 00:00:00')
    days_since_ref = (time_array_shifted - ref_date) / pd.Timedelta(days=1)

    # Duplicate dataset and assign shifted time
    filei_da_shifted = filei_da.copy(deep=True)
    filei_da_shifted = filei_da_shifted.assign_coords(time=('time', days_since_ref.astype('float32')))

    # Ensure time attributes are preserved
    filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
    filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'


    # Overwrite time as float array
    filei_da_shifted = filei_da_shifted.assign_coords(time=("time", days_since_ref.values.astype('float32')))
    filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
    filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'

    # Step 3: Add a new 'history' entry
    original_history = filei_da_shifted.attrs.get('history', '')
    new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2011–2015 data to 2021–2025 for LBC case study"
    combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
    filei_da_shifted.attrs['history'] = combined_history

    # Step 4: Shift date variable
    date_strings = filei_da_shifted['date'].values.astype(str)
    date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
    date_shifted = date_datetimes + pd.DateOffset(years=10)
    new_date_values = date_shifted.strftime('%Y%m%d').astype('int32')
    filei_da_shifted = filei_da_shifted.drop_vars('date')
    filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

    ### Step 5: Save to new NetCDF file
    fileOUTpath = f'{target_diri}{newfilename}'
    filei_da_shifted.to_netcdf(fileOUTpath)
    print("Saved to:", fileOUTpath)


Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_DMS_bb_surface_to2025use2011to2015_ne30np4_c20200606.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_num_so4_a1_bb_surface_to2025use2011to2015_ne30np4_c20200606.nc


In [49]:
# Other files
### Files to modify
# copy 2015 to 2022 got aircraft vertical
# - 'bc_a4 -> emissions-cmip6_bc_a4_aircraft_vertical_ 
# - 'NO2 -> emissions-cmip6_NO2_aircraft_vertical_   
# - 'num_a4 -> emissions-cmip6_num_bc_a4_aircraft_vertical_
# - 'SO2 -> emissions-cmip6_SO2_aircraft_vertical_

sourceCopy_diri = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/CMIP6_emissions_1750_2015/'
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

filels = [
    'emissions-cmip6_bc_a4_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc',
    'emissions-cmip6_NO2_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc',
    'emissions-cmip6_num_bc_a4_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc',
    'emissions-cmip6_SO2_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc',
         ]


In [50]:
# Loop through => except for time in cft format
cftimefiles = []
for fileidx in range(len(filels)):
    # Original file: CMIP6 emissions 
    filename = filels[fileidx]
    # print(filename)
    
    # New filename to reflect copying 2011–2015 data to 2021–2025
    newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

    # Define year range to extract
    startDate = '2011-01-01'
    endDate = '2015-12-31'

    try:
        # Read original file for years 2011–2015
        filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

        ### Process
        # Step 1: Create new time coordinates by adding 10 years (shift 2011–2015 → 2021–2025)
        time_array_pd = pd.to_datetime(filei_da['time'].values)
        time_array_shifted = time_array_pd + pd.DateOffset(years=10)
        time_array_shifted_np = time_array_shifted.to_numpy()

        # Step 2: Duplicate the dataset and assign shifted time as float days since 1750-01-01
        filei_da_shifted = filei_da.copy(deep=True)

        # Reference date for new time encoding
        ref_date = pd.Timestamp('1750-01-01 00:00:00')
        # Shifted datetime
        time_array_shifted = pd.to_datetime(filei_da['time'].values) + pd.DateOffset(years=10)
        # Convert to float days since 1750-01-01
        days_since_ref = (time_array_shifted - ref_date) / pd.Timedelta(days=1)

        # Overwrite time as float array
        filei_da_shifted = filei_da_shifted.assign_coords(time=("time", days_since_ref.values.astype('float32')))
        filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
        filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'

        # Step 3: Add a new 'history' entry
        original_history = filei_da_shifted.attrs.get('history', '')
        new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2011–2015 data to 2021–2025 for LBC case study"
        combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
        filei_da_shifted.attrs['history'] = combined_history

        # Step 4: Shift date variable
        date_strings = filei_da_shifted['date'].values.astype(str)
        date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
        date_shifted = date_datetimes + pd.DateOffset(years=10)
        new_date_values = date_shifted.strftime('%Y%m%d').astype('int32')
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        # Replace the 'date' variable
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        ### Step 5: Save to new NetCDF file
        fileOUTpath = f'{target_diri}{newfilename}'
        filei_da_shifted.to_netcdf(fileOUTpath)
        print("Saved to:", fileOUTpath)
    
    except TypeError:
        cftimefiles.append(filename)
        print(f'cftime - {filename}')


emissions-cmip6_bc_a4_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
emissions-cmip6_NO2_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_NO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
emissions-cmip6_num_bc_a4_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_num_bc_a4_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc
emissions-cmip6_SO2_aircraft_vertical_1750-2015_0.9x1.25_c20170608.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_SO2_aircraft_verticto2025use2011to2015_.9x1.25_c20170608.nc


In [51]:
cftimefiles

[]

In [58]:
### two ene files

sourceCopy_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4/'
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

filels = [
    'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_num_so4_a1_ene_vertical_c20210423.nc',
    'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_so4_a1_ene_vertical_c20210423.nc',
         ]


In [65]:
# Loop through => except for time in cft format
cftimefiles = []
for fileidx in range(len(filels)):
    # Original file: CMIP6 emissions 
    filename = filels[fileidx]
    # print(filename)
    
    # New filename to reflect copying 2018–2015 data to 2021–2025
    newfilename = filename

    # Define year range to extract
    startDate = '2019-01-01'
    endDate = '2021-12-31'

    try:
        # Read original file for years 2019–2021
        filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

        ### Process
        # Step 1: Create new time coordinates by adding 2 years (shift 2019–2021 → 2021–2023)
        shiftyear = 2
        time_array_pd = pd.to_datetime(filei_da['time'].values)
        time_array_shifted = time_array_pd + pd.DateOffset(years=shiftyear)
        time_array_shifted_np = time_array_shifted.to_numpy()

        # Step 2: Duplicate the dataset and assign shifted time as float days since 1750-01-01
        filei_da_shifted = filei_da.copy(deep=True)

        # Reference date for new time encoding
        ref_date = pd.Timestamp('1750-01-01 00:00:00')
        # Shifted datetime
        time_array_shifted = pd.to_datetime(filei_da['time'].values) + pd.DateOffset(years=shiftyear)
        # Convert to float days since 1750-01-01
        days_since_ref = (time_array_shifted - ref_date) / pd.Timedelta(days=1)

        # Overwrite time as float array
        filei_da_shifted = filei_da_shifted.assign_coords(time=("time", days_since_ref.values.astype('float32')))
        filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
        filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'

        # Step 3: Add a new 'history' entry
        original_history = filei_da_shifted.attrs.get('history', '')
        new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2019–2021 to 2021–2023 for case study"
        combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
        filei_da_shifted.attrs['history'] = combined_history

        # Step 4: Shift date variable
        date_strings = filei_da_shifted['date'].values.astype(str)
        date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
        date_shifted = date_datetimes + pd.DateOffset(years=shiftyear)
        new_date_values = date_shifted.strftime('%Y%m%d').astype('int32')
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        # Replace the 'date' variable
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        ### Step 5: Save to new NetCDF file
        fileOUTpath = f'{target_diri}{newfilename}'
        filei_da_shifted.to_netcdf(fileOUTpath)
        print("Saved to:", fileOUTpath)
    
    except TypeError: 
        cftimefiles.append(filename)
        print(f'cftime - {filename}')


Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_num_so4_a1_ene_vertical_c20210423.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_so4_a1_ene_vertical_c20210423.nc


### Draft below

In [34]:
testfile = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/emissions-cmip6_DMS_other_surface_1750_2015_ne30np4_c20200605.nc'

In [35]:
import cftime
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime, timedelta

ds = xr.open_dataset(testfile)

if isinstance(ds['time'].values[0], cftime.DatetimeNoLeap):
    # Step 1: Get attributes
    units = ds['time'].attrs.get('units', 'days since 1750-01-01 00:00:00')
    calendar = ds['time'].attrs.get('calendar', 'noleap')
    time_vals = ds['time'].values

    # Step 2: Convert to datetime using cftime (then shift)
    time_dt = np.array([pd.Timestamp(t.strftime('%Y-%m-%d')) for t in time_vals])
    time_dt_shifted = time_dt + pd.DateOffset(years=10)

    # Step 3: Convert shifted datetime back to float days since reference
    ref_date = datetime.strptime(units.split('since')[1].strip(), '%Y-%m-%d %H:%M:%S')
    shifted_days = np.array([(t - ref_date).days for t in time_dt_shifted])

    # Step 4: Assign new time (as float64) back to ds
    ds['time'] = ('time', shifted_days.astype('float64'))
    ds['time'].attrs['units'] = units
    ds['time'].attrs['calendar'] = calendar

else:
    print("Time is not in cftime format. Handle as datetime64 directly.")


Time is not in cftime format. Handle as datetime64 directly.


In [40]:
ds.time.values

array(['1750-01-16T00:00:00.000000000', '1750-02-16T00:00:00.000000000',
       '1750-03-16T00:00:00.000000000', ...,
       '2015-10-16T00:00:00.000000000', '2015-11-16T00:00:00.000000000',
       '2015-12-16T00:00:00.000000000'], dtype='datetime64[ns]')

In [44]:
# Loop through => except for time in cft format
cftimefiles = []
for fileidx in range(len(filels)):
    # Original file: CMIP6 emissions 
    filename = filels[fileidx]
    print(filename)
    
    # New filename to reflect copying 2011–2015 data to 2021–2025
    newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

    # Define year range to extract
    startDate = '2011-01-01'
    endDate = '2015-12-31'

    try:
        # Read original file for years 2011–2015
        filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

        ### Process
        # Step 1: Create new time coordinates by adding 10 years (shift 2011–2015 → 2021–2025)
        time_array_pd = pd.to_datetime(filei_da['time'].values)
        time_array_shifted = time_array_pd + pd.DateOffset(years=10)
        time_array_shifted_np = time_array_shifted.to_numpy()

        # Step 2: Duplicate the dataset and assign shifted time as float days since 1750-01-01
        filei_da_shifted = filei_da.copy(deep=True)

        # Reference date for new time encoding
        ref_date = pd.Timestamp('1750-01-01 00:00:00')
        # Shifted datetime
        time_array_shifted = pd.to_datetime(filei_da['time'].values) + pd.DateOffset(years=10)
        # Convert to float days since 1750-01-01
        days_since_ref = (time_array_shifted - ref_date) / pd.Timedelta(days=1)

        # Overwrite time as float array
        filei_da_shifted = filei_da_shifted.assign_coords(time=("time", days_since_ref.values.astype('float32')))
        filei_da_shifted['time'].attrs['units'] = 'days since 1750-01-01 00:00:00'
        filei_da_shifted['time'].attrs['calendar'] = 'Gregorian'

        # Step 3: Add a new 'history' entry
        original_history = filei_da_shifted.attrs.get('history', '')
        new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2011–2015 data to 2021–2025 for LBC case study"
        combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
        filei_da_shifted.attrs['history'] = combined_history

        # Step 4: Shift date variable
        date_strings = filei_da_shifted['date'].values.astype(str)
        date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
        date_shifted = date_datetimes + pd.DateOffset(years=10)
        new_date_values = date_shifted.strftime('%Y%m%d').astype('int32')
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        # Replace the 'date' variable
        filei_da_shifted = filei_da_shifted.drop_vars('date')
        filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

        ### Step 5: Save to new NetCDF file
        fileOUTpath = f'{target_diri}{newfilename}'
        filei_da_shifted.to_netcdf(fileOUTpath)
        print("Saved to:", fileOUTpath)
    
    except TypeError:
        cftimefiles.append(filename)
        print(f'cftime - {filename}')


emissions-cmip6_DMS_bb_surface_175001-201512_ne30np4_c20200606.nc
cftime - emissions-cmip6_DMS_bb_surface_175001-201512_ne30np4_c20200606.nc
emissions-cmip6_DMS_other_surface_1750_2015_ne30np4_c20200605.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_DMS_other_surfto2025use2011to2015_ne30np4_c20200605.nc
emissions-cmip6_num_so4_a1_bb_surface_175001-201512_ne30np4_c20200606.nc
cftime - emissions-cmip6_num_so4_a1_bb_surface_175001-201512_ne30np4_c20200606.nc


In [43]:
testfile = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/emissions-cmip6_DMS_other_surface_1750_2015_ne30np4_c20200605.nc'
filei_da = xr.open_dataset(testfile)
filei_da

<xarray.Dataset> Size: 622MB
Dimensions:       (time: 3192, ncol: 48602)
Coordinates:
  * time          (time) datetime64[ns] 26kB 1750-01-16 ... 2015-12-16
Dimensions without coordinates: ncol
Data variables:
    lon           (ncol) float64 389kB ...
    lat           (ncol) float64 389kB ...
    area          (ncol) float64 389kB ...
    emiss_oceans  (time, ncol) float32 621MB ...
    date          (time) int32 13kB ...
Attributes: (12/16)
    data_reference:    Used in CESM1 for CCMI, e.g., Tilmes et al., GMD, 2016.
    data_source_url:   /glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/ccm...
    data_script:       /glade/u/home/emmons/CMIP6/emissions/make_emis_cmip6_o...
    history:           A single year of emissions (monthly variation) is repe...
    update_date:       20170322
    creation_date:     20170322
    ...                ...
    data_title:        Natural emissions of DMS from CCMI for CMIP6
    interp:             FV Source data interpolated to SE grid using ESMF 
    source_file:       emissions-cmip6_DMS_other_surface_1750-2015_0.9x1.25_c...
    interp_method:     conserve
    srcAttBegin:       ---SOURCE-FILE-ATTRIBUTES-------------------
    srcAttEnd:         ---SOURCE-FILE-ATTRIBUTES-------------------

In [45]:
filei_da_shifted

<xarray.Dataset> Size: 13MB
Dimensions:       (ncol: 48602, time: 60)
Coordinates:
  * time          (time) float32 240B 9.9e+04 9.903e+04 ... 1.008e+05 1.008e+05
Dimensions without coordinates: ncol
Data variables:
    lon           (ncol) float64 389kB ...
    lat           (ncol) float64 389kB ...
    area          (ncol) float64 389kB ...
    emiss_oceans  (time, ncol) float32 12MB ...
    date          (time) int32 240B 20210116 20210216 ... 20251116 20251216
Attributes: (12/16)
    data_reference:    Used in CESM1 for CCMI, e.g., Tilmes et al., GMD, 2016.
    data_source_url:   /glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/ccm...
    data_script:       /glade/u/home/emmons/CMIP6/emissions/make_emis_cmip6_o...
    history:           A single year of emissions (monthly variation) is repe...
    update_date:       20170322
    creation_date:     20170322
    ...                ...
    data_title:        Natural emissions of DMS from CCMI for CMIP6
    interp:             FV Source data interpolated to SE grid using ESMF 
    source_file:       emissions-cmip6_DMS_other_surface_1750-2015_0.9x1.25_c...
    interp_method:     conserve
    srcAttBegin:       ---SOURCE-FILE-ATTRIBUTES-------------------
    srcAttEnd:         ---SOURCE-FILE-ATTRIBUTES-------------------

In [29]:
# Loop through => ONLY for time in cft format
sourceCopy_diri = '/net/fs01/data/cesm2/inputdata/atm/cam/chem/emis/historical_ne30np4/'
target_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_intermediate/'

from datetime import datetime
import xarray as xr
import pandas as pd
import os

for fileidx in range(len(cftimefiles)):
    filename = cftimefiles[fileidx]
    
    # New filename reflects 10-year shift
    newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

    # Read in file
    filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

    ### Step 1: Shift time coordinate by 10 years (for 2011–2015 → 2021–2025)
    time_cftime = filei_da['time'].values
    time_datetime = pd.to_datetime([t.isoformat() for t in time_cftime])
    time_array_shifted = pd.to_datetime([
        t.replace(year=t.year + 10) if 2011 <= t.year <= 2015 else t
        for t in time_datetime
    ])
    filei_da_shifted = filei_da.copy(deep=True)
    filei_da_shifted['time'] = time_array_shifted.to_numpy()

    ### Step 2: Update 'date' variable accordingly
    date_strings = filei_da_shifted.date.values.astype(str)
    date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
    date_shifted = date_datetimes + pd.DateOffset(years=10)
    new_date_values = date_shifted.strftime('%Y%m%d').astype(int)

    filei_da_shifted = filei_da_shifted.drop_vars('date')
    filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

    ### Step 3: Add history entry
    original_history = filei_da_shifted.attrs.get('history', '')
    new_history_entry = (
        f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: "
        f"Copied data from 2011–2015 to 2021–2025 for LBC case study."
    )
    combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
    filei_da_shifted.attrs['history'] = combined_history

    ### Step 4: Save to output path
    fileOUTpath = f'{target_diri}{newfilename}'
    filei_da_shifted.to_netcdf(fileOUTpath)
    print("Saved to:", fileOUTpath)


Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_intermediate/emissions-cmip6_DMS_bb_surface_to2025use2011to2015_ne30np4_c20200606.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_intermediate/emissions-cmip6_num_so4_a1_bb_surface_to2025use2011to2015_ne30np4_c20200606.nc


In [31]:
# # convert file format to classic
# input_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_intermediate/'
# output_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

# # Ensure the output directory exists
# os.makedirs(output_dir, exist_ok=True)

# # Convert each .nc file
# for filename in os.listdir(input_dir):
#     if filename.endswith('.nc'):
#         input_file_path = os.path.join(input_dir, filename)
#         output_file_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.nc")
        
#         # Open the NetCDF-4 file
#         with Dataset(input_file_path, 'r') as src:
#             # Create a new NetCDF classic file
#             with Dataset(output_file_path, 'w', format='NETCDF3_CLASSIC') as dst:
#                 # Copy dimensions
#                 for name, dimension in src.dimensions.items():
#                     dst.createDimension(name, (len(dimension) if not dimension.isunlimited() else None))

#                 # Copy variables
#                 for name, variable in src.variables.items():
#                     dst.createVariable(name, variable.datatype, variable.dimensions)
#                     dst[name][:] = src[name][:]
#                     dst[name].setncatts({k: variable.getncattr(k) for k in variable.ncattrs()})

#         print(f"Converted {input_file_path} to {output_file_path}")


In [ ]:
# => time in dt64 format

In [16]:
### One file

from datetime import datetime
import pandas as pd
import xarray as xr

# Original file: CMIP6 emissions for DMS_other_surface
filename = 'emissions-cmip6_DMS_other_surface_1750_2015_ne30np4_c20200605.nc'

# New filename to reflect copying 2011–2015 data to 2021–2025
newfilename = f'{filename[:-34]}to2025use2011to2015_{filename[-20:]}'

# Define year range to extract
startDate = '2011-01-01'
endDate = '2015-12-31'

# Read original file for years 2011–2015
filei_da = xr.open_dataset(sourceCopy_diri + filename, engine='netcdf4').sel(time=slice(startDate, endDate))

### Process
# Step 1: Create new time coordinates by adding 10 years (shift 2011–2015 → 2021–2025)
time_array_pd = pd.to_datetime(filei_da['time'].values)
time_array_shifted = time_array_pd + pd.DateOffset(years=10)
time_array_shifted_np = time_array_shifted.to_numpy()

# Step 2: Duplicate the dataset and assign shifted time
filei_da_shifted = filei_da.copy(deep=True)
filei_da_shifted['time'] = time_array_shifted_np

# Step 3: Add a new 'history' entry
original_history = filei_da_shifted.attrs.get('history', '')
new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Copied 2011–2015 data to 2021–2025 for LBC case study"
combined_history = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry
filei_da_shifted.attrs['history'] = combined_history

# Step 4: Shift the 'date' variable by 10 years (assuming it's formatted as YYYYMMDD integers)
date_strings = filei_da_shifted['date'].values.astype(str)
date_datetimes = pd.to_datetime(date_strings, format='%Y%m%d')
date_shifted = date_datetimes + pd.DateOffset(years=10)
new_date_values = date_shifted.strftime('%Y%m%d').astype(int)

# Replace the 'date' variable
filei_da_shifted = filei_da_shifted.drop_vars('date')
filei_da_shifted = filei_da_shifted.assign(date=('time', new_date_values))

### Step 5: Save to new NetCDF file
fileOUTpath = f'{target_diri}{newfilename}'
filei_da_shifted.to_netcdf(fileOUTpath)
print("Saved to:", fileOUTpath)


Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_netCDF4/emissions-cmip6_DMS_other_surfto2025use2011to2015_ne30np4_c20200605.nc


In [18]:
# import os
# from netCDF4 import Dataset

# # Directories
# input_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_netCDF4/'
# output_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/'

# os.makedirs(output_dir, exist_ok=True)

# for filename in os.listdir(input_dir):
#     if filename.endswith('.nc'):
#         input_file_path = os.path.join(input_dir, filename)
#         output_file_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.nc")

#         with Dataset(input_file_path, 'r') as src:
#             with Dataset(output_file_path, 'w', format='NETCDF3_CLASSIC') as dst:
#                 # Copy dimensions
#                 for name, dimension in src.dimensions.items():
#                     dst.createDimension(name, (len(dimension) if not dimension.isunlimited() else None))

#                 # Copy variables
#                 for name, variable in src.variables.items():
#                     dtype = variable.datatype
#                     if dtype == 'int64' or str(dtype).startswith('datetime'):
#                         print(f"Skipping variable '{name}' (unsupported dtype {dtype} in NETCDF3_CLASSIC)")
#                         continue  # or cast to int32 or float64 if appropriate
#                     dst_var = dst.createVariable(name, dtype, variable.dimensions)
#                     dst_var[:] = variable[:]
#                     dst_var.setncatts({k: variable.getncattr(k) for k in variable.ncattrs()})

#                 # Copy global attributes
#                 dst.setncatts({k: src.getncattr(k) for k in src.ncattrs()})

#         print(f"Converted {input_file_path} to {output_file_path}")


Skipping variable 'time' (unsupported dtype int64 in NETCDF3_CLASSIC)
Skipping variable 'date' (unsupported dtype int64 in NETCDF3_CLASSIC)
Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023_netCDF4/emissions-cmip6_DMS_other_surfto2025use2011to2015_ne30np4_c20200605.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/approxY2022Y2023/emissions-cmip6_DMS_other_surfto2025use2011to2015_ne30np4_c20200605.nc


# Modify nl file

In [7]:
### Get spc list
CAMS_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/'

# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(CAMS_diri, 'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_*.nc')

# Get list of matching files in full path
CAMS_v51_file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
CAMS_v51_file_names = [os.path.basename(f) for f in CAMS_v51_file_list]

# print(CAMS_v51_file_names[:3])

import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'ne30np4_(.+?)_c20210423', f).group(1)
    for f in CAMS_v51_file_names
]
species_names = sorted(species_names)
print(species_names)
print(len(species_names))

['BENZENE', 'BIGALK', 'BIGENE', 'C2H2', 'C2H4', 'C2H5OH', 'C2H6', 'C3H6', 'C3H8', 'CH2O', 'CH3CHO', 'CH3CN', 'CH3COCH3', 'CH3COOH', 'CH3OH', 'CO', 'HCN', 'HCOOH', 'HONO', 'ISOP', 'IVOC', 'MEK', 'MTERP', 'NH3', 'NO', 'SO2', 'SVOC', 'TOLUENE', 'XYLENES', 'bc_a4', 'num_bc_a4', 'num_pom_a4', 'num_so4_a1_ag', 'num_so4_a1_ene_vertical', 'num_so4_a1_ship_fef', 'num_so4_a2_res_trs', 'pom_a4', 'so4_a1_ag_ship_fef', 'so4_a1_ene_vertical', 'so4_a2_res_trs']
40


In [8]:
import os

# Define the old and new base paths
old_base_path = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/'
new_base_path = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/'

# Input and output file paths
input_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/Working_nl_copies/work2_user_nl_cam'
output_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/test05_user_nl_cam'

# Read and update the lines in the input file
updated_lines = []
with open(input_file_path, 'r') as f:
    for line in f:
        for spc in species_names:
            old_filename = f"CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc"
            new_filename = f"CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc"
            old_full_path = os.path.join(old_base_path, old_filename)
            new_full_path = os.path.join(new_base_path, new_filename)

            if old_full_path in line:
                if os.path.isfile(new_full_path):
                    line = line.replace(old_full_path, new_full_path)
                else:
                    print(f'{new_full_path} not available to replace')
        updated_lines.append(line)

# Write the updated lines to the output file
with open(output_file_path, 'w') as f:
    f.writelines(updated_lines)

print('File updated and saved:', output_file_path)


File updated and saved: /home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/test05_user_nl_cam


In [2]:
import os

# Define the old and new base paths
old_base_path = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/'
new_base_path = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4/'


# Input and output file paths
input_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/test01_user_nl_cam'
output_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/test02_user_nl_cam'

# Read and update the lines in the input file
updated_lines = []
with open(input_file_path, 'r') as f:
    for line in f:
        for spc in species_names:
            old_filename = f"CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc"
            new_filename = f"CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc"
            old_full_path = os.path.join(old_base_path, old_filename)
            new_full_path = os.path.join(new_base_path, new_filename)

            if old_full_path in line:
                if os.path.isfile(new_full_path):
                    line = line.replace(old_full_path, new_full_path)
                else:
                    print(f'{new_full_path} not available to replace')
        updated_lines.append(line)

# Write the updated lines to the output file
with open(output_file_path, 'w') as f:
    f.writelines(updated_lines)

print('File updated and saved:', output_file_path)


File updated and saved: /home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/test02_user_nl_cam


In [11]:
# When Remove BB emissions files
### Read in original emissions data
original_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/qfed2.6_finn/ne30np4/'
CONUSlandMasked_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/qfed2.6_finn/ne30np4_CONUSlandMasked_80kmBuffer/'

In [12]:
# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(original_diri, 'qfed.emis_*_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc')

# Get list of matching files in full path
file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
file_names = [os.path.basename(f) for f in file_list]

print(file_names[:3])

import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'qfed.emis_(.+?)_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc', f).group(1)
    for f in file_names
]
species_names = sorted(species_names)
print(species_names)

['qfed.emis_SO2_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc', 'qfed.emis_BIGALK_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc', 'qfed.emis_MVK_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc']
['BC', 'BENZENE', 'BIGALK', 'BIGENE', 'C2H2', 'C2H4', 'C2H5OH', 'C2H6', 'C3H6', 'C3H8', 'CH2O', 'CH3CHO', 'CH3CN', 'CH3COCH3', 'CH3COCHO', 'CH3COOH', 'CH3OH', 'CH4', 'CO', 'CO2', 'CRESOL', 'GLYALD', 'HCN', 'HCOOH', 'HYAC', 'ISOP', 'IVOC', 'MACR', 'MEK', 'MVK', 'NH3', 'NO', 'NO2', 'OC', 'SO2', 'SVOC', 'TOLUENE', 'XYLENES', 'bc_a4', 'num_bc_a4', 'num_pom_a4', 'pom_a4']


In [13]:
import os

# Define the old and new base paths
old_base_path = original_diri
new_base_path = CONUSlandMasked_diri

# Input and output file paths
input_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.BASEY20220401TY20230401_user_nl_cam'
output_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.noBBemisCONUS80kmBuffer_user_nl_cam'

# Read and update the lines in the input file
updated_lines = []
with open(input_file_path, 'r') as f:
    for line in f:
        for spc in species_names:
            old_filename = f"qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc"
            new_filename = f"qfed.emis_{spc}_bb_surface_daily_20171201T20231231_ne30np4_mol_c20240126.nc"
            old_full_path = os.path.join(old_base_path, old_filename)
            new_full_path = os.path.join(new_base_path, new_filename)

            if old_full_path in line:
                if os.path.isfile(new_full_path):
                    line = line.replace(old_full_path, new_full_path)
                else:
                    print(f'{new_full_path} not available to replace')
        updated_lines.append(line)

# Write the updated lines to the output file
with open(output_file_path, 'w') as f:
    f.writelines(updated_lines)

print('File updated and saved:', output_file_path)


File updated and saved: /home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.noBBemisCONUS80kmBuffer_user_nl_cam


In [5]:
# When Remove ANTHRO emissions files
### Read in original emissions data
original_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/'
CONUSlandMasked_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_CONUSlandMasked_80kmBuffer/'

In [9]:
# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(original_diri, 'CAMS-GLOB-ANT_ne30np4_*_v6.2_monthly.nc')

# Get list of matching files in full path
file_list = glob.glob(pattern)

# Extract only the filenames
file_names = [os.path.basename(f) for f in file_list]

# print(file_names[:3])

import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'CAMS-GLOB-ANT_ne30np4_(.+?)_v6.2_monthly.nc', f).group(1)
    for f in file_names
]
species_names = sorted(species_names)
print(species_names)

['BENZENE', 'BIGALK', 'BIGENE', 'C2H2', 'C2H4', 'C2H5OH', 'C2H6', 'C3H6', 'C3H8', 'CH2O', 'CH3CHO', 'CH3CN', 'CH3COCH3', 'CH3COOH', 'CH3OH', 'CO', 'HCN', 'HCOOH', 'ISOP', 'IVOC', 'MEK', 'MTERP', 'NH3', 'NO', 'SO2', 'SVOC', 'TOLUENE', 'XYLENES', 'bc_a4', 'num_bc_a4', 'num_pom_a4', 'num_so4_a1_ag', 'num_so4_a1_ene_vertical', 'num_so4_a1_ship_fef', 'num_so4_a2_res_trs', 'pom_a4', 'so4_a1_ag_ship_fef', 'so4_a1_ene_vertical', 'so4_a2_res_trs']


In [10]:
import os

# Define the old and new base paths
old_base_path = original_diri
new_base_path = CONUSlandMasked_diri

# Input and output file paths
input_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.BASEY20220401TY20230401_user_nl_cam'
output_file_path = '/home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.noANTHROemisCONUS80kmBuffer_user_nl_cam'

# Read and update the lines in the input file
updated_lines = []
with open(input_file_path, 'r') as f:
    for line in f:
        for spc in species_names:
            old_filename = f"CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc"
            new_filename = f"CAMS-GLOB-ANT_ne30np4_{spc}_v6.2_monthly.nc"
            old_full_path = os.path.join(old_base_path, old_filename)
            new_full_path = os.path.join(new_base_path, new_filename)

            if old_full_path in line:
                if os.path.isfile(new_full_path):
                    line = line.replace(old_full_path, new_full_path)
                else:
                    print(f'{new_full_path} not available to replace')
        updated_lines.append(line)

# Write the updated lines to the output file
with open(output_file_path, 'w') as f:
    f.writelines(updated_lines)

print('File updated and saved:', output_file_path)


File updated and saved: /home/taoma528/Scripts/CESM_analysis/CONUS_BackgroundO3/BGO3.noANTHROemisCONUS80kmBuffer_user_nl_cam


# Given the trouble with those two vertical emissions files
now try to use CAMS5.1 for those

In [47]:
# Problem child
# '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_num_so4_a1_ene_vertical_c20210423_modified.nc'
# '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_so4_a1_ene_vertical_c20210423_modified.nc'
updated_spcs = ['num_so4_a1_ene_vertical','so4_a1_ene_vertical',]

ANTsource_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4/'
ANTtarget_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/'
# ANTtarget_diri = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/'

In [33]:
### ANT - Copy 2021 to 2023

# Define the start and end dates
startDate = '2019-01-01'
endDate = '2022-01-01'

def safe_shift_year(t, shift=2):
    try:
        return t.replace(year=t.year + shift)
    except ValueError:
        # For Feb 29 → shift using DateOffset to handle leap year issues
        return t + pd.DateOffset(years=shift)

In [48]:
for SPCidx, SPCi in enumerate(updated_spcs):
    # Define input file name and path
    SPCi_filename = f'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{SPCi}_c20210423.nc'
    SPCi_INpath = f'{ANTsource_diri}{SPCi_filename}'
    
    # Open and select data from 2019–2021
    SPCi_da = xr.open_dataset(SPCi_INpath, engine='netcdf4').sel(time=slice(startDate, endDate))

    ### Step 1: Shift time coordinates by 2 years (2019→2021, 2020→2022, 2021→2023)
    time_array_pd = pd.to_datetime(SPCi_da.time.values)
    time_array_shifted = pd.to_datetime([
        safe_shift_year(t, shift=2) if t.year in [2019, 2020, 2021] else t
        for t in time_array_pd
    ])
    time_array_shifted_np = time_array_shifted.to_numpy()

    ### Step 2: Duplicate dataset and assign shifted time
    SPCi_da_shifted = SPCi_da.copy(deep=True)
    SPCi_da_shifted['time'] = time_array_shifted_np

    ### Step 3: Shift the 'date' variable if it exists
    if 'date' in SPCi_da_shifted:
        date_vals = SPCi_da_shifted['date'].values.astype(str)
        date_dt = pd.to_datetime(date_vals, format='%Y%m%d')
        date_shifted = date_dt + pd.DateOffset(years=2)
        date_new_vals = date_shifted.strftime('%Y%m%d').astype(np.int32)

        SPCi_da_shifted = SPCi_da_shifted.drop_vars('date')
        SPCi_da_shifted = SPCi_da_shifted.assign(date=('time', date_new_vals))

    ### Step 4: Extract coordinate/static variables
    lat = SPCi_da['lat']
    lon = SPCi_da['lon']
    area = SPCi_da['area']

    ### Step 5: Concatenate original and shifted data
    data_vars = [var for var in SPCi_da.data_vars if var not in ['lat', 'lon', 'area']]
    combined_da = xr.concat([SPCi_da[data_vars], SPCi_da_shifted[data_vars]], dim='time')

    ### Step 6: Restore static fields
    combined_da = combined_da.assign(lat=lat, lon=lon, area=area)

    ### Step 7: Preserve metadata and encoding
    combined_da.attrs = SPCi_da.attrs
    for var in combined_da.data_vars:
        combined_da[var].attrs = SPCi_da[var].attrs
        combined_da[var].encoding = SPCi_da[var].encoding

    ### Step 8: Add updated history
    original_history = SPCi_da.attrs.get('history', '')
    new_history_entry = f"{datetime.now().strftime('%a %b %d %H:%M:%S %Y')}: Duplicated 2019–2021 as 2021–2023 and updated 'date' variable"
    combined_da.attrs['history'] = f"{original_history}\n{new_history_entry}" if original_history else new_history_entry

    ### Step 9: Save to new NetCDF file
    newSPCi_filename = SPCi_filename.replace('CAMS-GLOB-ANT_v5.1_2000-2021', 'CAMS-GLOB-ANT_v5.1_2019-2023repeated')
    fileOUTpath = f'{ANTtarget_diri}{newSPCi_filename}'
    combined_da.to_netcdf(fileOUTpath)
    print("Saved to:", fileOUTpath)


Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_num_so4_a1_ene_vertical_c20210423.nc
Saved to: /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_so4_a1_ene_vertical_c20210423.nc


In [46]:
combined_da

<xarray.Dataset> Size: 226MB
Dimensions:   (time: 72, altitude: 8, ncol: 48602)
Coordinates:
  * altitude  (altitude) float64 64B 0.025 0.075 0.125 ... 0.275 0.325 0.375
  * ncol      (ncol) float64 389kB 0.0 1.0 2.0 ... 4.86e+04 4.86e+04 4.86e+04
  * time      (time) datetime64[ns] 576B 2019-01-01 2019-02-01 ... 2023-11-30
Data variables:
    date      (time) int32 288B 20190115 20190215 20190315 ... 20231015 20231115
    ene       (time, altitude, ncol) float32 112MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ind       (time, altitude, ncol) float32 112MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    lat       (ncol) float64 389kB ...
    lon       (ncol) float64 389kB ...
    area      (ncol) float64 389kB ...
Attributes: (12/28)
    distribution_date:                    2021-02-22 16:26:36 UTC
    Conventions:                          CF-1.6
    projection:                           latlon
    authors:                              N.Elguindi and C.Granier
    product_version:                      v4.2-S1.1
    title:                                CAMS Global anthropogenic emissions
    ...                                   ...
    remmaped_from_files:                  from file(s): /glade/p/nsc/nacd0023...
    mapping_equation:                     so4_a1_ene_vertical[96]        =   ...
    remapped_by:                          Species_Mapping.py tool
    remapping_time:                       2021-04-23 14:09:37.912867
    remmaping_username:                   cdswk on crhtc36.hpc.ucar.edu
    comment_from_Species_Mapping_script:  day of each month of date array was...

In [35]:
# for SPCidx,SPCi in enumerate(updated_spcs):
#     # print(SPCidx,SPC)
    
#     # Filter files that contain both SPC and StartStr
#     SPCi_filename = f'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{SPCi}_c20210423.nc'
    
#     # Read in one file as an example
#     SPCi_INpath = f'{ANTsource_diri}{SPCi_filename}'
#     SPCi_da = xr.open_dataset(SPCi_INpath, engine='netcdf4').sel(time=slice(startDate, endDate))

#     ### Process
#     # Step 1: Create new time coordinates for 2022
#     time_array_2021 = SPCi_da.time.values
#     # Replace the year 2021 with 2022
#     # Convert the numpy datetime64 array to a pandas DatetimeIndex
#     time_array_pd = pd.to_datetime(time_array_2021)

#     # Replace 2019 with 2021, 2020 with 2022, 2021 with 2023 using a list comprehension | apply the defined function
#     time_array_2022 = pd.to_datetime([
#         safe_shift_year(t, shift=2) if t.year in [2019, 2020, 2021] else t
#         for t in time_array_pd
#     ])

#     # Convert back to numpy datetime64 array
#     time_array_2022_np = time_array_2022.to_numpy()

#     # Step 2: Duplicate the original dataset for the new time period
#     SPCi_da_2022 = SPCi_da.copy(deep=True)
#     SPCi_da_2022['time'] = time_array_2022_np
    
#     # Step 3: Extract the unchanged variables
#     lat = SPCi_da['lat']
#     lon = SPCi_da['lon']
#     area = SPCi_da['area']

#     # Step 4: Concatenate the original 2021 data with the duplicated 2022 data
#     # Concatenate only the varying data variables (excluding 'lat', 'lon', and 'area')
#     data_vars = [var for var in SPCi_da.data_vars if var not in ['lat', 'lon', 'area']]
#     combined_da = xr.concat([SPCi_da[data_vars], SPCi_da_2022[data_vars]], dim='time')

#     # Step 5: Reassign the unchanged variables back to the combined dataset
#     combined_da = combined_da.assign(lat=lat, lon=lon, area=area)

#     # Step 6: Ensure attributes, encoding, and everything else are retained
#     combined_da.attrs = SPCi_da.attrs
#     for var in combined_da.data_vars:
#         combined_da[var].attrs = SPCi_da[var].attrs
#         combined_da[var].encoding = SPCi_da[var].encoding

#     # Step 7: Add the history to document changes
#     # Get the current history attribute from the original dataset
#     original_history = SPCi_da.attrs.get('history', '')

#     ### Save to a new NetCDF file
#     # Replace the start of the string
#     newSPCi_filename = SPCi_filename.replace('CAMS-GLOB-ANT_v5.1_2000-2021', 'CAMS-GLOB-ANT_v5.1_2021-2022repeated')

#     fileOUTpath = f'{ANTtarget_diri}{newSPCi_filename}'
#     combined_da.to_netcdf(fileOUTpath)
#     print("Save to: ",fileOUTpath)
    
#     # # Define encoding for classic NetCDF format
#     # encoding = {var: {'format': 'NETCDF3_CLASSIC'} for var in combined_da.data_vars}
#     # fileOUTpath = f'{ANTtarget_diri}{newSPCi_filename}'
#     # combined_da.to_netcdf(fileOUTpath, engine='netcdf4', encoding=encoding)
#     # print("Save to: ",fileOUTpath)

Save to:  /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/CAMS-GLOB-ANT_v5.1_2021-2022repeated_ne30np4_num_so4_a1_ene_vertical_c20210423.nc
Save to:  /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/CAMS-GLOB-ANT_v5.1_2021-2022repeated_ne30np4_so4_a1_ene_vertical_c20210423.nc


In [39]:
time_array_2022_np

array(['2021-01-01T00:00:00.000000000', '2021-02-01T00:00:00.000000000',
       '2021-03-01T00:00:00.000000000', '2021-04-01T00:00:00.000000000',
       '2021-05-01T00:00:00.000000000', '2021-06-01T00:00:00.000000000',
       '2021-07-01T00:00:00.000000000', '2021-08-01T00:00:00.000000000',
       '2021-09-01T00:00:00.000000000', '2021-10-01T00:00:00.000000000',
       '2021-11-01T00:00:00.000000000', '2021-12-01T00:00:00.000000000',
       '2022-01-01T00:00:00.000000000', '2022-02-01T00:00:00.000000000',
       '2022-02-28T00:00:00.000000000', '2022-03-31T00:00:00.000000000',
       '2022-04-30T00:00:00.000000000', '2022-05-31T00:00:00.000000000',
       '2022-06-30T00:00:00.000000000', '2022-07-31T00:00:00.000000000',
       '2022-08-31T00:00:00.000000000', '2022-09-30T00:00:00.000000000',
       '2022-10-31T00:00:00.000000000', '2022-11-30T00:00:00.000000000',
       '2022-12-31T00:00:00.000000000', '2023-01-31T00:00:00.000000000',
       '2023-02-28T00:00:00.000000000', '2023-03-31

In [40]:
combined_da

<xarray.Dataset> Size: 226MB
Dimensions:   (time: 72, altitude: 8, ncol: 48602)
Coordinates:
  * altitude  (altitude) float64 64B 0.025 0.075 0.125 ... 0.275 0.325 0.375
  * ncol      (ncol) float64 389kB 0.0 1.0 2.0 ... 4.86e+04 4.86e+04 4.86e+04
  * time      (time) datetime64[ns] 576B 2019-01-01 2019-02-01 ... 2023-11-30
Data variables:
    date      (time) int32 288B 20190115 20190215 20190315 ... 20211015 20211115
    ene       (time, altitude, ncol) float32 112MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ind       (time, altitude, ncol) float32 112MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    lat       (ncol) float64 389kB ...
    lon       (ncol) float64 389kB ...
    area      (ncol) float64 389kB ...
Attributes: (12/28)
    distribution_date:                    2021-02-22 16:26:36 UTC
    Conventions:                          CF-1.6
    projection:                           latlon
    authors:                              N.Elguindi and C.Granier
    product_version:                      v4.2-S1.1
    title:                                CAMS Global anthropogenic emissions
    ...                                   ...
    remmaped_from_files:                  from file(s): /glade/p/nsc/nacd0023...
    mapping_equation:                     so4_a1_ene_vertical[96]        =   ...
    remapped_by:                          Species_Mapping.py tool
    remapping_time:                       2021-04-23 14:09:37.912867
    remmaping_username:                   cdswk on crhtc36.hpc.ucar.edu
    comment_from_Species_Mapping_script:  day of each month of date array was...

In [49]:
import xarray as xr
from netCDF4 import Dataset

In [50]:
### Convert format
import os
from netCDF4 import Dataset

# Directory containing NetCDF-4 files
input_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/'
output_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/'

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Convert each .nc file
for filename in os.listdir(input_dir):
    if filename.endswith('.nc'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.nc")
        
        # Open the NetCDF-4 file
        with Dataset(input_file_path, 'r') as src:
            # Create a new NetCDF classic file
            with Dataset(output_file_path, 'w', format='NETCDF3_CLASSIC') as dst:
                # Copy dimensions
                for name, dimension in src.dimensions.items():
                    dst.createDimension(name, (len(dimension) if not dimension.isunlimited() else None))

                # Copy variables
                for name, variable in src.variables.items():
                    dst.createVariable(name, variable.datatype, variable.dimensions)
                    dst[name][:] = src[name][:]
                    dst[name].setncatts({k: variable.getncattr(k) for k in variable.ncattrs()})

        print(f"Converted {input_file_path} to {output_file_path}")


Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_num_so4_a1_ene_vertical_c20210423.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_num_so4_a1_ene_vertical_c20210423.nc
Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021_netCDF4/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_so4_a1_ene_vertical_c20210423.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y20222023_copied20202021/CAMS-GLOB-ANT_v5.1_2019-2023repeated_ne30np4_so4_a1_ene_vertical_c20210423.nc


In [51]:
### Convert format
import os
from netCDF4 import Dataset

# Directory containing NetCDF-4 files
input_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/'
output_dir = '/net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_NETCDF3/'

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Convert each .nc file
for filename in os.listdir(input_dir):
    if filename.endswith('.nc'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.nc")
        
        # Open the NetCDF-4 file
        with Dataset(input_file_path, 'r') as src:
            # Create a new NetCDF classic file
            with Dataset(output_file_path, 'w', format='NETCDF3_CLASSIC') as dst:
                # Copy dimensions
                for name, dimension in src.dimensions.items():
                    dst.createDimension(name, (len(dimension) if not dimension.isunlimited() else None))

                # Copy variables
                for name, variable in src.variables.items():
                    dst.createVariable(name, variable.datatype, variable.dimensions)
                    dst[name][:] = src[name][:]
                    dst[name].setncatts({k: variable.getncattr(k) for k in variable.ncattrs()})

        print(f"Converted {input_file_path} to {output_file_path}")


Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/CAMS-GLOB-ANT_ne30np4_CH3COCH3_v6.2_monthly.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_NETCDF3/CAMS-GLOB-ANT_ne30np4_CH3COCH3_v6.2_monthly.nc
Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/CAMS-GLOB-ANT_ne30np4_num_so4_a2_res_trs_v6.2_monthly.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_NETCDF3/CAMS-GLOB-ANT_ne30np4_num_so4_a2_res_trs_v6.2_monthly.nc
Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/nonNANne30np4/CAMS-GLOB-ANT_ne30np4_C3H6_v6.2_monthly.nc to /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANT_v6.2/ne30np4_NETCDF3/CAMS-GLOB-ANT_ne30np4_C3H6_v6.2_monthly.nc
Converted /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cam

In [ ]:
# /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_num_so4_a1_ene_vertical_c20210423.nc
# /net/fs09/d0/taoma528/cheyenne_copies/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_num_so4_a1_ene_vertical_c20210423_modified.nc

In [ ]:
import xarray as xr
from netCDF4 import Dataset